# 只用 NumPy 手写线性回归

线性回归可以看作最简单的神经网络：只有一个没有激活函数的神经元。

- 前向计算：$\hat{y} = Xw + b$
- 损失函数：$L = \frac{1}{n}\sum(\hat{y}-y)^2$
- 反向传播：手动计算 $w$ 和 $b$ 的梯度
- 参数更新：$w \leftarrow w-\eta\frac{\partial L}{\partial w}$，$b \leftarrow b-\eta\frac{\partial L}{\partial b}$

下面的训练部分只依赖 NumPy，不使用 PyTorch、TensorFlow 或自动求导。

In [1]:
import numpy as np

# 使用固定随机种子，保证每次运行都得到相同结果
rng = np.random.default_rng(42)

# 制造一批带噪声的数据：y = 2.5x - 0.8 + noise
num_samples = 200
true_w = 2.5
true_b = -0.8

X_np = rng.uniform(-3.0, 3.0, size=(num_samples, 1))
noise = rng.normal(0.0, 0.1, size=(num_samples, 1))
y_np = true_w * X_np + true_b + noise

print('X shape:', X_np.shape)
print('y shape:', y_np.shape)
print('前 5 个样本：')
print(np.hstack((X_np[:5], y_np[:5])))

X shape: (200, 1)
y shape: (200, 1)
前 5 个样本：
[[ 1.64373629  3.37367341]
 [-0.36672936 -1.75628392]
 [ 2.15158752  4.57845661]
 [ 1.18420817  2.14417615]
 [-2.43493591 -6.85358233]]


In [2]:
def linear_forward(X, w, b):
    """前向计算：根据当前参数得到预测值。"""
    return X @ w + b


def mse_loss(y_hat, y):
    """均方误差：预测值和真实值之间的平均平方距离。"""
    return np.mean((y_hat - y) ** 2)


def backward(X, y_hat, y):
    """手动计算均方误差对 w 和 b 的梯度。"""
    num_samples = X.shape[0]
    error = y_hat - y

    # dL/dw = (2/n) * X.T @ (y_hat - y)
    dw = (2.0 / num_samples) * (X.T @ error)
    # dL/db = (2/n) * sum(y_hat - y)
    db = (2.0 / num_samples) * np.sum(error, axis=0)
    return dw, db

In [8]:
# 1. 初始化可学习参数
w_np = rng.normal(0.0, 0.1, size=(1, 1))
b_np = np.zeros(1)

# 2. 设置训练超参数
learning_rate = 0.01
num_epochs = 500
loss_history = []

# 3. 训练：前向计算 -> 计算损失 -> 反向传播 -> 更新参数
for epoch in range(1, num_epochs + 1):
    y_hat = linear_forward(X_np, w_np, b_np)
    loss = mse_loss(y_hat, y_np)
    dw, db = backward(X_np, y_hat, y_np)

    w_np -= learning_rate * dw
    b_np -= learning_rate * db
    loss_history.append(loss)

    if epoch == 1 or epoch % 50 == 0:
        print(f'epoch {epoch:3d} | loss = {loss:.6f}')
        print(f'w = {w_np.item():.3f}, b = {b_np.item():.3f}')

print(f'\n真实参数：w = {true_w:.3f}, b = {true_b:.3f}')
print(f'学习结果：w = {w_np.item():.3f}, b = {b_np.item():.3f}')

# 用训练好的模型预测新数据
X_new = np.array([[-2.0], [0.0], [2.0]])
predictions = linear_forward(X_new, w_np, b_np)
print('\n新数据预测：')
for x, prediction in zip(X_new.ravel(), predictions.ravel()):
    print(f'x = {x:4.1f} -> y_hat = {prediction:6.3f}')

epoch   1 | loss = 18.520293
w = 0.163, b = -0.018
epoch  50 | loss = 0.143600
w = 2.384, b = -0.523
epoch 100 | loss = 0.020396
w = 2.506, b = -0.698
epoch 150 | loss = 0.011324
w = 2.511, b = -0.762
epoch 200 | loss = 0.010139
w = 2.511, b = -0.785
epoch 250 | loss = 0.009982
w = 2.511, b = -0.793
epoch 300 | loss = 0.009961
w = 2.511, b = -0.796
epoch 350 | loss = 0.009959
w = 2.511, b = -0.797
epoch 400 | loss = 0.009958
w = 2.511, b = -0.798
epoch 450 | loss = 0.009958
w = 2.511, b = -0.798
epoch 500 | loss = 0.009958
w = 2.511, b = -0.798

真实参数：w = 2.500, b = -0.800
学习结果：w = 2.511, b = -0.798

新数据预测：
x = -2.0 -> y_hat = -5.820
x =  0.0 -> y_hat = -0.798
x =  2.0 -> y_hat =  4.224
